# Глава 5. Предварительное обучение на неразмеченных данных

In [ ]:
pip install matplotlib numpy tiktoken torch tensorflow

In [ ]:
from importlib.metadata import version

pkgs = ["matplotlib", 
        "numpy", 
        "tiktoken", 
        "torch",
        "tensorflow" # Для предварительно обученных моделей OpenAI
       ]
for p in pkgs:
    print(f"{p} Версия: {version(p)}")

- В этой главе мы реализуем цикл обучения и код для базовой оценки модели, чтобы провести предварительное обучение большой языковой модели
- В конце мы также загружаем в нашу модель общедоступные предварительно обученные веса от OpenAI

<img src="https://camo.githubusercontent.com/137f57f6192fbcb6627e6ced1b5274c71924774dec18a4dea29b1c156619ef24/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30312e77656270" width=800px>

- Ниже перечислены темы, затронутые в этой главе

<img src="https://camo.githubusercontent.com/01ebc99e37dddc617ba6dd20799f945fd6a562bac8b8abe5a4b82eb491ebbfca/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30322e77656270" width=800px>

&nbsp;
## 5.1 Оценка генеративных текстовых моделей

- В начале этого раздела мы кратко расскажем о том, как инициализировать модель GPT с помощью кода из предыдущей главы
- Затем мы обсудим основные метрики оценки больших языковых моделей
- Наконец, в этом разделе мы применим эти метрики оценки к обучающему и проверочному наборам данных

&nbsp;
### 5.1.1 Использование GPT для генерации текста

- Мы инициализируем модель GPT с помощью кода из предыдущей главы

In [ ]:
import torch
from previous_chapters import GPTModel
# Если файл `previous_chapters.py` недоступен локально,
# вы можете импортировать его из пакета PyPI `llms-from-scratch`. 
# Подробнее см.: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# Например,
# from llms_from_scratch.ch04 import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Размер словаря
    "context_length": 256, # Сокращенная длина контекста (исходное значение: 1024)
    "emb_dim": 768,        # Размерность эмбеддинга
    "n_heads": 12,         # Количество ядер внимания
    "n_layers": 12,        # Количество слоев
    "drop_rate": 0.1,      # Коэффициент дропаута
    "qkv_bias": False      # Смещение в сторону значений ключей запроса
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();  # Отключите дропаут во время логического вывода

- Мы используем дропаут 0,1, но в настоящее время довольно часто обучают большие языковые модели без дропаута
- В современных больших языковых моделях также не используются векторы смещения в слоях `nn.Linear` для матриц запросов, ключей и значений (в отличие от более ранних моделей GPT). Это достигается за счет установки параметра `"qkv_bias": False`
- Мы уменьшили длину контекста (`context_length`) всего на 256 токенов, чтобы снизить требования к вычислительным ресурсам для обучения модели, в то время как исходная модель GPT-2 со 124 миллионами параметров использовала 1024 токена
    - Это сделано для того, чтобы мы могли следить за примерами кода и выполнять их на своих портативных компьютерах
    - Позже мы также загрузим модель с `context_length` 1024 из предварительно обученных весов.

- Далее мы используем функцию `generate_text_simple` из предыдущей главы для генерации текста
- Кроме того, мы определяем две вспомогательные функции: `text_to_token_ids` и `token_ids_to_text` — для преобразования токенов в текстовое представление и обратно, которые мы будем использовать на протяжении всей главы

<img src="https://camo.githubusercontent.com/0756c65e7e7878cab7b7673bedd3f441cf42f1f67e89b46d9e7ec931e0fffbc3/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30332e77656270" width=800px>

In [ ]:
import tiktoken
from previous_chapters import generate_text_simple

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # добавить размер пакета
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # удалить размер пакета
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Выводимый текст:\n", token_ids_to_text(token_ids, tokenizer))

- Как мы видим выше, модель не генерирует качественный текст, потому что она еще не обучена
- Как измерить или зафиксировать в числовом выражении, что такое «качественный текст», чтобы отслеживать этот показатель во время обучения?
- В следующем подразделе мы рассмотрим метрики для расчета показателя потерь для сгенерированных результатов, которые можно использовать для оценки прогресса обучения
- В следующих главах, посвященных тонкой настройке больших языковых моделей, мы также рассмотрим дополнительные способы оценки качества модели

&nbsp;
### 5.1.2 Расчет потерь при генерации текста: кросс-энтропия и перплексия

- Предположим, у нас есть тензор `inputs`, содержащий идентификаторы токенов для двух обучающих примеров (строк)
- Соответствующие `inputs`, `targets` содержат желаемые идентификаторы токенов, которые мы хотим, чтобы модель сгенерировала
- Обратите внимание, что `targets` — это `inputs`, сдвинутые на одну позицию

In [ ]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",Ф
                        [1107,  588, 11311]]) #  " really like chocolate"]

- Подавая на вход модели данные, мы получаем вектор логитов для двух входных примеров, каждый из которых состоит из 3 токенов
- Каждый токен представляет собой вектор из 50 257 элементов, соответствующий размеру словаря
- Применяя функцию softmax, мы можем преобразовать тензор логитов в тензор той же размерности, содержащий оценки вероятности

In [ ]:
with torch.no_grad():
    logits = model(inputs)

probas = torch.softmax(logits, dim=-1) # Вероятность появления каждого токена в словаре
print(probas.shape) # Форма: (размер пакета, количество токенов, размер словаря)

- На рисунке ниже с использованием очень небольшого набора слов для наглядности показано, как мы преобразуем оценки вероятности обратно в текст

<img src="https://camo.githubusercontent.com/f98790fc96dfefdd3e61533ca406f241a893976c8cb7668afbcec178763c45a0/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30342e77656270" width=800px>

- Ммы можем использовать функцию `argmax`, чтобы преобразовать значения вероятности в предсказанные идентификаторы токенов
- Функция `softmax`, описанная выше, сгенерировала 50 257-мерный вектор для каждого токена. Функция `argmax` возвращает позицию с наибольшим значением вероятности в этом векторе, которая и является предсказанным идентификатором токена

- Поскольку у нас есть 2 входных пакета по 3 токена в каждом, мы получаем 2 на 3 предсказанных идентификатора токенов:

In [ ]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print("Идентификаторы токенов:\n", token_ids)

- Если мы расшифруем эти токены, то увидим, что они сильно отличаются от тех, которые мы хотим, чтобы модель предсказывала

In [ ]:
print(f"Целевой batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Фактический batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

- Это потому, что модель еще не обучена
- Чтобы обучить модель, нам нужно знать, насколько она далека от правильных прогнозов (целевых значений)

<img src="https://camo.githubusercontent.com/3dee5bf33ad015c683fa2a9a91ae611b263101027fa2f9415934ae1a1c38778e/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30362e77656270" width=800px>

- Вероятности появления токенов, соответствующие целевым индексам, следующие:

In [ ]:
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Текст 1:", target_probas_1)

text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Текст 2:", target_probas_2)

 ---

### Код выше

Эта строка выполняет **индексирование многомерного массива (тензора) `probas`** для извлечения вероятностей, соответствующих правильным целевым классам.

### Пошаговый разбор

### 1. `text_idx`
Скалярный индекс, указывающий на конкретный текст в батче.
- `probas[text_idx, ...]` → выбирает двумерный срез формы `(количество_токенов, количество_классов)` для одного текста.

### 2. `[0, 1, 2]`
Список индексов токенов.
- `probas[text_idx, [0, 1, 2], ...]` → выбирает строки только для токенов с индексами 0, 1, 2 (первые три токена).
- После этого измерения получается форма `(3, количество_классов)`.

### 3. `targets[text_idx]`
Одномерный массив формы `(количество_токенов,)`, содержащий **истинные метки классов** для каждого токена в тексте `text_idx`.
- `targets[text_idx]` → вектор правильных классов для всего текста.
- Но поскольку на предыдущем шаге мы выбрали только токены `[0, 1, 2]`, здесь **неявно тоже берутся первые три элемента** этого вектора (благодаря broadcasting/advanced indexing).

### 4. Полное индексирование
```python
probas[text_idx, [0, 1, 2], targets[text_idx]]
```
NumPy/PyTorch выполняет **advanced indexing**:
- Для каждого из выбранных токенов (0, 1, 2) извлекается вероятность того класса, который указан в `targets` для этого же токена.
- Фактически это эквивалентно:
```python
[
    probas[text_idx, 0, targets[text_idx][0]],  # вер-ть правильного класса для токена 0
    probas[text_idx, 1, targets[text_idx][1]],  # вер-ть правильного класса для токена 1
    probas[text_idx, 2, targets[text_idx][2]]   # вер-ть правильного класса для токена 2
]
```

### Результат

**`target_probas_2`** — одномерный массив из трёх чисел (вероятностей), показывающих, насколько модель была уверена в **правильных** классах для первых трёх токенов конкретного текста.

Это часто используется для:
- анализа уверенности модели в правильных ответах,
- вычисления **confidence** срезов,
- поиска сложных примеров (где верная вероятность мала),
- отладки качества предсказаний на уровне отдельных токенов.

---

- Мы хотим максимизировать все эти значения, приблизив их к вероятности 1.
- В математической оптимизации проще максимизировать логарифм показателя вероятности, чем сам показатель вероятности. Лекция с более подробным описанием: [L8.2 Функция потерь логистической регрессии](https://www.youtube.com/watch?v=GxJe0DZvydM)

In [ ]:
# Вычислить логарифм всех вероятностей токенов
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(log_probas)

- Далее мы вычисляем среднюю логарифмическую вероятность:

In [ ]:
# Рассчитайте среднюю вероятность для каждого токена
avg_log_probas = torch.mean(log_probas)
print(avg_log_probas)

- Цель состоит в том, чтобы сделать эту среднюю логарифмическую вероятность как можно более высокой за счет оптимизации весовых коэффициентов модели
- Из-за логарифмической функции максимально возможное значение равно 0, а мы пока далеки от этого значения

- В глубоком обучении вместо максимизации средней логарифмической вероятности принято минимизировать *отрицательное* значение средней логарифмической вероятности. В нашем случае вместо того, чтобы максимизировать -10.7940, чтобы оно приблизилось к 0, в глубоком обучении мы минимизируем -10.7940, чтобы оно приблизилось к 0
- Отрицательное значение -10.7940, то есть -10.7940, в глубоком обучении также называют кросс-энтропийной потерей

In [ ]:
neg_avg_log_probas = avg_log_probas * -1
print(neg_avg_log_probas)

В PyTorch уже реализована функция `cross_entropy`, которая выполняет описанные выше действия

<img src="https://camo.githubusercontent.com/3d6fb2ee91bebb246351570506a344d7d579fc161084faec6856c0659c815452/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30372e77656270" width=800px>

- Прежде чем применить функцию `cross_entropy`, давайте проверим форму логитов и целевых значений

In [ ]:
# Логиты имеют форму (batch_size, num_tokens, vocab_size)
print("Форма логитов:", logits.shape)

# Цель имеет форму (batch_size, num_tokens)
print("Целевая форма:", targets.shape)

- Для функции `cross_entropy` в PyTorch мы хотим сгладить эти тензоры, объединив их по размерности пакета:

In [ ]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()

print("Сглаженные логиты:", logits_flat.shape)
print("Сглаженные цели:", targets_flat.shape)

- Обратите внимание, что целевыми значениями являются идентификаторы токенов, которые также представляют собой позиции индексов в тензорах логитов, которые мы хотим максимизировать
- Функция `cross_entropy` в PyTorch автоматически применяет функцию `softmax` и вычисляет логарифмическую вероятность для тех индексов токенов в логитах, которые нужно максимизировать

In [ ]:
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

- Понятие, связанное с кросс-энтропийной потерей, — это перплексия большой языковой модели
- Перплексия — это просто экспоненциальная функция от кросс-энтропийной потери

In [ ]:
perplexity = torch.exp(loss)
print(perplexity)

- Показатель перплексии часто считается более интерпретируемым, поскольку его можно рассматривать как эффективный размер словаря, в котором модель не уверена на каждом этапе (в приведенном выше примере это 48 725 слов или токенов)
- Другими словами, перплексия показывает, насколько хорошо распределение вероятностей, предсказанное моделью, соответствует реальному распределению слов в наборе данных
- Как и в случае с логарифмической функцией потерь, чем ниже перплексия, тем ближе предсказания модели к реальному распределению

 ---

### Представим, что большая языковая модель — это **попугай**, который учится говорить.

### 🦜 1. Зачем нужна модель (попугай)
Представь, что у нас есть попугай, который пока не умеет говорить. Мы хотим научить его заканчивать фразы. Мы говорим: **"Каждое усилие двигает..."**, а попугай должен договорить: **"...тебя"**.

Но сначала попугай даже не знает слов и говорит полную чушь: **"Каждое усилие двигает крокодил летать фиолетовый"**.

### 📏 2. Как измерить, насколько плох попугай? (кросс-энтропия)
Нам нужна линейка, чтобы **измерить ошибки** попугая.

### Шаг 1: Попугай выдаёт вероятности
Когда мы говорим слово, попугай-модель не выдаёт сразу одно слово — он выдаёт **список всех слов** с их вероятностями. Например:
- "тебя" — вероятность 0.0001 (очень маленькая, он почти не верит в это слово)
- "крокодил" — вероятность 0.8 (он очень верит, что дальше будет "крокодил")

### Шаг 2: Смотрим, какую вероятность попугай дал ПРАВИЛЬНОМУ слову
Мы знаем правильное слово — "тебя". Попугай дал ему вероятность 0.0001. Это **очень плохо**. Если бы он был умным, он дал бы вероятность 0.9999 (почти 1).

Код:
```python
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
```
**Перевод:** "Попугай, покажи, насколько ты веришь в правильные слова 'effort', 'moves', 'you'. Ага, веришь на 0.0001, 0.0002, 0.00005 — ужасно!"

### 🔢 3. Почему берут логарифм?
Числа вроде 0.0000001 неудобно складывать и сравнивать. **Логарифм** — это как специальное увеличительное стекло, которое превращает очень маленькие вероятности в "нормальные" отрицательные числа:
- 0.0001 → логарифм = -9.21
- 0.5 → логарифм = -0.69
- 0.999 → логарифм = -0.001

Чем ближе к 0, тем лучше. У идеального попугая логарифмы были бы 0.

### ✖️ 4. Зачем переворачивать знак? (кросс-энтропия)
В школе нас учат **минимизировать ошибки**, а не максимизировать успех. Поэтому мы берём логарифмы и **умножаем на -1**, чтобы перевернуть:
- Было: цель — чтобы логарифм стал 0 (максимум).
- Стало: цель — чтобы **отрицательный логарифм** стал 0 (минимум).

`neg_avg_log_probas = avg_log_probas * -1` — это как сказать: "Твоя ошибка сейчас 10.794. Уменьшай её до 0!"

Это число и есть **кросс-энтропийная потеря** — главная оценка, насколько сильно ошибается попугай.

### 🤯 5. Что такое перплексия? (простыми словами)
**Перплексия = exp(потеря)**.

Это число говорит: **"Попугай сейчас как будто выбирает из скольких слов?"**

- Если перплексия = **48,725** (как в примере) → попугай в панике, как будто перед ним словарь из 48 тысяч слов, и он гадает наугад.
- Если перплексия = **5** → попугай уже почти выучился, колеблется только между 5 похожими словами.
- Если перплексия = **1** → попугай идеально знает каждое слово.

### 🎯 Итог: зачем ВСЁ ЭТО?
Весь этот сложный код с вероятностями, логарифмами и перплексией нужен для одного:

**Чтобы компьютер мог САМ, без человека, понять, хорошо ли он учится говорить.**
- Смотрит на правильные слова
- Смотрит, какие вероятности он им дал
- Считает "ошибку" (кросс-энтропию)
- Старается уменьшить ошибку → учится говорить лучше

Это как если бы попугай сам себя проверял по учебнику и исправлял ошибки, пока не заговорит, как человек.

---

&nbsp;
### 5.1.3 Расчет потерь для обучающей и проверочной выборок

- Для обучения большой языковой модели мы используем относительно небольшой набор данных (по сути, всего одну короткую историю)
- Причины в следующем:
  - Вы можете запустить примеры кода за несколько минут на ноутбуке без подходящего графического процессора
  - Обучение завершается относительно быстро (за несколько минут, а не недель), что удобно для образовательных целей
  - Мы используем текст из общественного достояния, который можно включить в этот репозиторий на GitHub без нарушения авторских прав и без увеличения размера репозитория


- Например, для обучения Llama 2 7B на 2 триллионах токенов потребовалось 184 320 часов работы на графических процессорах A100
  - На момент написания этой статьи почасовая стоимость облачного сервера 8xA100 на AWS составляла примерно 30 долларов США
  - Таким образом, по приблизительным подсчетам, обучение этой большой языковой модели обойдется в 184 320 / 8 * 30 долларов США = 690 000 долларов США

In [ ]:
import os
import requests

file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()


# Изначально в книге использовался следующий код:
# Однако urllib использует более старые настройки протокола, которые
# могут вызвать проблемы у некоторых пользователей VPN. 
# Приведенная выше версия с использованием requests более надежна
# в этом отношении.

        
# import os
# import urllib.request

# file_path = "the-verdict.txt"
# url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

# if not os.path.exists(file_path):
#     with urllib.request.urlopen(url) as response:
#         text_data = response.read().decode('utf-8')
#     with open(file_path, "w", encoding="utf-8") as file:
#         file.write(text_data)
# else:
#     with open(file_path, "r", encoding="utf-8") as file:
#         text_data = file.read()

- Быстрая проверка корректности загрузки текста путем вывода на экран первых и последних 99 символов

In [ ]:
# Первые 99 символов
print(text_data[:99])

In [ ]:
# Последние 99 символов
print(text_data[-99:])

In [ ]:
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))

print("Количество символов:", total_characters)
print("Количество токенов:", total_tokens)

- Текст состоит из 5145 токенов и слишком короток для обучения большой языковой модели, но, повторюсь, это делается в образовательных целях (позже мы также загрузим предварительно обученные модели)

- Далее мы делим набор данных на обучающую и проверочную выборки и с помощью загрузчиков данных из главы 2 подготавливаем пакеты для обучения большой языковой модели
- Для наглядности на рисунке ниже указано значение `max_length=6`, но для загрузчика обучающей выборки мы устанавливаем `max_length` равным длине контекста, поддерживаемой большой языковой моделью
- Для простоты на рисунке показаны только входные токены
    - Поскольку мы обучаем LLM предсказывать следующее слово в тексте, цели выглядят так же, как и эти входные данные, за исключением того, что цели сдвинуты на одну позицию

<img src="https://camo.githubusercontent.com/216592059168cdb0fba2d8b9843bfc411a1898c407163cabb1d98a59e7cc584a/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30392e77656270" width=800px>

In [ ]:
from previous_chapters import create_dataloader_v1

# Соотношение обучения и валидации
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [ ]:
# Проверка на вменяемость

if total_tokens * (train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Недостаточно токенов для загрузчика обучения. "
          "Попробуйте уменьшить `GPT_CONFIG_124M['context_length']` или "
          "увеличить `training_ratio`")

if total_tokens * (1-train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Недостаточно токенов для валидационного загрузчика. "
          "Попробуйте уменьшить `GPT_CONFIG_124M['context_length']` или "
          "снизить `training_ratio`")

- Мы используем относительно небольшой размер пакета, чтобы снизить нагрузку на вычислительные ресурсы, а также потому, что исходный набор данных очень мал
- Например, Llama 2 7B обучалась с размером пакета 1024

- Дополнительная проверка правильности загрузки данных:

In [ ]:
print("Обучающий загрузчик:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nПроверочный загрузчик:")
for x, y in val_loader:
    print(x.shape, y.shape)

- Еще одна дополнительная проверка, подтверждающая, что размер токенов соответствует ожидаемому:

In [ ]:
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

print("Обучающие токены:", train_tokens)
print("Проверочные токены:", val_tokens)
print("Все токены:", train_tokens + val_tokens)

- Далее мы реализуем вспомогательную функцию для расчета кросс-энтропийной потери для заданного пакета данных
- Кроме того, мы реализуем вторую вспомогательную функцию для расчета потерь для заданного пользователем количества пакетов в загрузчике данных

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Уменьшите количество пакетов до общего количества пакетов в загрузчике данных
        # если num_batches превышает количество пакетов в загрузчике данных
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

 ---

### Код выше

Представь, что ты учишь робота предсказывать следующее слово в тексте. Эти две функции — как учитель, который проверяет, насколько хорошо робот справляется с заданием.

### Первая функция: `calc_loss_batch` — "Проверка одной пачки заданий"

```python
def calc_loss_batch(input_batch, target_batch, model, device):
    # Перемещаем данные на "рабочий стол" (GPU или CPU)
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    
    # Даём роботу входные данные, он выдаёт свои догадки
    logits = model(input_batch)
    
    # Сравниваем догадки робота с правильными ответами
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    
    return loss  # Возвращаем "количество ошибок"
```

**Что происходит простыми словами:**
- Ты даёшь роботу **несколько примеров сразу** (пачку), чтобы он работал быстрее
- Робот смотрит на начало фразы и пытается угадать следующее слово
- Функция сравнивает догадки робота с правильными ответами
- Чем больше ошибок — тем больше число `loss` (потеря)
- Если робот угадал идеально — `loss` будет около 0

**Зачем нужно:**
- Чтобы понять, насколько хорошо робот учится
- Если `loss` уменьшается — робот становится умнее!

### Вторая функция: `calc_loss_loader` — "Проверка всей домашней работы"

```python
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.  # Корзинка для сбора всех ошибок
    
    # Если нет ни одного задания — говорим "не могу посчитать"
    if len(data_loader) == 0:
        return float("nan")
    
    # Сколько пачек заданий проверить?
    elif num_batches is None:
        num_batches = len(data_loader)  # Проверим все пачки
    else:
        num_batches = min(num_batches, len(data_loader))  # Проверим сколько просят
    
    # Проходим по пачкам заданий
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            # Проверяем очередную пачку и добавляем ошибки в корзинку
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()  # .item() превращает в обычное число
        else:
            break  # Проверили сколько нужно — останавливаемся
    
    # Возвращаем среднее количество ошибок на пачку
    return total_loss / num_batches
```

**Что происходит простыми словами:**
- Представь, что у тебя есть большая стопка тетрадей (это `data_loader`)
- Ты можешь проверить все тетради или только часть (это `num_batches`)
- Ты берёшь каждую тетрадь, проверяешь её (вызываешь первую функцию) и записываешь сколько ошибок
- Потом считаешь **среднее количество ошибок** на одну тетрадь

**Почему именно так:**

1. **Проверка пачками (batch), а не по одному примеру:**
   - Это быстрее! Как проверять сразу несколько тетрадей, а не по одной
   - Компьютер хорошо умеет делать много одинаковых действий параллельно

2. **Можно проверить не всё (`num_batches`):**
   - Иногда данных очень много, проверка всего занимает часы
   - Можно проверить только часть, чтобы быстро понять, как идёт обучение

3. **`float("nan")` если данных нет:**
   - Это как сказать "невозможно посчитать", если тетрадей вообще нет
   - Защита от ошибки деления на ноль

4. **`loss.item()` вместо просто `loss`:**
   - PyTorch хранит числа в специальной "умной" упаковке
   - `.item()` достаёт простое число, которое можно складывать в корзинку

5. **Среднее значение в конце (`total_loss / num_batches`):**
   - Честное сравнение: неважно, проверили мы 10 пачек или 100
   - Всегда получаем среднюю ошибку на одну пачку
   
### Простая аналогия: 

Представь, что ты учишь попугая говорить:

- **Первая функция** — это когда ты показываешь попугаю 32 слова и проверяешь, сколько из них он повторил неправильно
- **Вторая функция** — это когда ты проверяешь его целый день (или час), записываешь все ошибки, а потом считаешь "в среднем 5 ошибок за раз"

Если с каждым днём среднее количество ошибок уменьшается — попугай учится! 🦜

 ---

- Если у вас есть компьютер с графическим процессором, поддерживающим CUDA, LLM будет обучаться на графическом процессоре без каких-либо изменений в коде
- С помощью параметра `device` мы гарантируем, что данные будут загружены на то же устройство, что и модель LLM

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Для стабильных результатов используйте PyTorch 2.9 или более новую версию
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print(f"Используется устройство: {device}.")

model.to(device)  # для классов nn.Module не требуется присваивание model = model.to(device)

torch.manual_seed(123)  # Для воспроизводимости результатов из-за перемешивания в загрузчике данных

with torch.no_grad():  # Отключаем отслеживание градиентов для эффективности, так как модель пока не обучается
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print("Потери на обучающей выборке:", train_loss)
print("Потери на проверочной выборке:", val_loss)

<img src="https://camo.githubusercontent.com/de4c90a17bebd1ed30b5ae1724724d2fb8fec8233258eb2c6c931e17fb3a3b5a/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f31302e77656270" width=800px>

&nbsp;
## 5.2 Обучение LLM

- В этом разделе мы наконец реализуем код для обучения большой языковой модели
- Мы сосредоточимся на простой функции обучения

<img src="https://camo.githubusercontent.com/37533967fe3f55ee41068b1d695b4664c81b2b4b228d369f52a77148c3c01682/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f31312e77656270" width=800px>

In [ ]:
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    # Инициализация списков для отслеживания потерь и просмотренных токенов
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    # Основной цикл обучения
    for epoch in range(num_epochs):
        model.train()  # Переводит модель в режим обучения
        
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # Сброс градиентов потерь с предыдущей итерации пакета
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # Вычисляет градиенты потерь
            optimizer.step() # Обновляет веса модели, используя градиенты потерь
            tokens_seen += input_batch.numel()
            global_step += 1

            # Дополнительный шаг оценки
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Эпоха {epoch+1} (Шаг {global_step:06d}): "
                      f"Потери на обучении {train_loss:.3f}, Потери на валидации {val_loss:.3f}")

        # Вывод образца текста после каждой эпохи
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()  # Переводит модель в режим оценки
    with torch.no_grad():  # Отключает вычисление градиентов
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()  # Возвращает модель в режим обучения
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()  # Переводит модель в режим оценки
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():  # Отключает вычисление градиентов
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # Компактный формат вывода
    model.train()  # Возвращает модель в режим обучения

- Теперь давайте обучим LLM с помощью функции обучения, описанной выше:

In [ ]:
import time
start_time = time.time()

torch.manual_seed(123)  # Фиксирует генератор случайных чисел для воспроизводимости
model = GPTModel(GPT_CONFIG_124M)
model.to(device)  # Перемещает модель на целевое устройство (GPU/CPU)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

num_epochs = 10
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Обучение завершено за {execution_time_minutes:.2f} минут.")

- Обратите внимание, что на вашем компьютере значения потерь могут немного отличаться. Это не повод для беспокойства, если они примерно одинаковы (потери при обучении ниже 1, а потери при проверке ниже 7)
. Небольшие различия часто могут быть связаны с разным оборудованием графического процессора и версиями CUDA, а также с небольшими изменениями в новых версиях PyTorch
- Даже если вы запускаете пример на центральном процессоре, вы можете заметить небольшие различия. Возможная причина — разное поведение nn.Dropout в разных операционных системах в зависимости от того, как был скомпилирован PyTorch, как обсуждалось [здесь, в системе отслеживания проблем PyTorch](https://github.com/pytorch/pytorch/issues/121595)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # Построение графиков потерь на обучении и валидации по эпохам
    ax1.plot(epochs_seen, train_losses, label="Потери на обучении")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Потери на валидации")
    ax1.set_xlabel("Эпохи")
    ax1.set_ylabel("Потери")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))  # Отображать только целочисленные метки на оси X

    # Создание второй оси X для отображения просмотренных токенов
    ax2 = ax1.twiny()  # Создаёт вторую ось X, использующую ту же ось Y
    ax2.plot(tokens_seen, train_losses, alpha=0)  # Невидимый график для выравнивания делений
    ax2.set_xlabel("Просмотренные токены")

    fig.tight_layout()  # Корректировка макета для размещения элементов
    plt.savefig("loss-plot.pdf")
    plt.show()

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

- Глядя на результаты выше, мы видим, что вначале модель генерирует бессвязные наборы слов, тогда как к концу она способна создавать грамматически более или менее правильные предложения
- Однако, основываясь на значениях потерь на обучающем и валидационном наборах, мы видим, что модель начинает переобучаться
- Если бы мы проверили несколько отрывков, которые она пишет ближе к концу, мы бы обнаружили, что они дословно содержатся в обучающем наборе — модель просто запоминает обучающие данные
- Позже мы рассмотрим стратегии декодирования, которые могут в определённой степени смягчить это запоминание
- Обратите внимание, что переобучение здесь происходит из-за того, что у нас очень и очень маленький обучающий набор, и мы проходим по нему так много раз
  - Обучение языковой модели здесь в первую очередь служит образовательным целям; мы главным образом хотим убедиться, что модель способна научиться генерировать связный текст
  - Вместо того чтобы тратить недели или месяцы на обучение этой модели на огромных объёмах данных с использованием дорогостоящего оборудования, позже мы загрузим предобученные веса

<img src="https://camo.githubusercontent.com/fc5d290513e57d77770122795090dc2455d3791cfc4f836cbec591a42c392a05/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f31332e77656270" width=800px>

&nbsp;
## 5.3 Стратегии декодирования для контроля случайности

- Инференс относительно дёшев с относительно небольшой LLM, такой как обученная нами выше модель GPT, поэтому нет необходимости использовать для него GPU, если вы использовали GPU для её обучения выше
- Используя функцию `generate_text_simple` (из предыдущей главы), которую мы использовали ранее внутри простой функции обучения, мы можем генерировать новый текст по одному слову (или токену) за раз
- Как объяснялось в разделе 5.1.2, следующий сгенерированный токен — это токен, соответствующий наибольшей вероятностной оценке среди всех токенов в словаре

In [ ]:
# НОВОЕ: используем CPU здесь, так как инференс дёшев с
# этой моделью и чтобы получать одинаковые результаты
inference_device = torch.device("cpu")

model.to(inference_device)
model.eval()

tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(inference_device),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

- Даже если мы выполним функцию `generate_text_simple` выше несколько раз, LLM всегда будет генерировать одни и те же выходные данные
- Теперь мы введём два понятия, так называемые стратегии декодирования, чтобы модифицировать `generate_text_simple`: *температурное масштабирование* и *top-k* сэмплирование
- Они позволят модели контролировать случайность и разнообразие генерируемого текста

&nbsp;
### 5.3.1 Масштабирование температуры

- Ранее мы всегда выбирали токен с наибольшей вероятностью в качестве следующего токена, используя `torch.argmax`
- Чтобы добавить разнообразие, мы можем выбирать следующий токен с помощью `torch.multinomial(probs, num_samples=1)`, осуществляя сэмплирование из вероятностного распределения
- Здесь шанс каждого индекса быть выбранным соответствует его вероятности во входном тензоре

- Вот небольшое повторение процесса генерации следующего токена, с использованием очень маленького словаря для иллюстративных целей:

In [ ]:
vocab = { 
    "closer": 0,
    "every": 1, 
    "effort": 2, 
    "forward": 3,
    "inches": 4,
    "moves": 5, 
    "pizza": 6,
    "toward": 7,
    "you": 8,
} 

inverse_vocab = {v: k for k, v in vocab.items()}

# Предположим, на вход подано "every effort moves you", и LLM
# возвращает следующие логиты для следующего токена:
next_token_logits = torch.tensor(
    [4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
)

probas = torch.softmax(next_token_logits, dim=0)
next_token_id = torch.argmax(probas).item()

# Затем следующий сгенерированный токен определяется так:
print(inverse_vocab[next_token_id])

In [ ]:
torch.manual_seed(123)
next_token_id = torch.multinomial(probas, num_samples=1).item()
print(inverse_vocab[next_token_id])

- Вместо определения наиболее вероятного токена через `torch.argmax`, мы используем `torch.multinomial(probas, num_samples=1)` для определения наиболее вероятного токена путём сэмплирования из softmax-распределения
- В иллюстративных целях давайте посмотрим, что происходит, когда мы сэмплируем следующий токен 1 000 раз, используя исходные softmax-вероятности:

In [ ]:
def print_sampled_tokens(probas):
    torch.manual_seed(123) # Ручная установка seed для воспроизводимости
    sample = [torch.multinomial(probas, num_samples=1).item() for i in range(1_000)]
    sampled_ids = torch.bincount(torch.tensor(sample), minlength=len(probas))
    for i, freq in enumerate(sampled_ids):
        print(f"{freq} x {inverse_vocab[i]}")

print_sampled_tokens(probas)

- Мы можем контролировать распределение и процесс выбора с помощью концепции, называемой температурным масштабированием
- «Температурное масштабирование» — это просто красивое название для деления логитов на число больше 0
- Температуры больше 1 приведут к более равномерно распределённым вероятностям токенов после применения softmax
- Температуры меньше 1 приведут к более уверенным (более острым или более пиковым) распределениям после применения softmax

- Обратите внимание, что результирующие выходные данные dropout могут выглядеть по-разному в зависимости от вашей операционной системы; вы можете подробнее прочитать об этой несогласованности [здесь, в трекере проблем PyTorch](https://github.com/pytorch/pytorch/issues/121595)

In [ ]:
def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)

# Значения температуры
temperatures = [1, 0.1, 5]  # Исходная, более высокая уверенность и более низкая уверенность

# Вычисляем масштабированные вероятности
scaled_probas = [softmax_with_temperature(next_token_logits, T) for T in temperatures]

In [ ]:
# Построение графика
x = torch.arange(len(vocab))
bar_width = 0.15

fig, ax = plt.subplots(figsize=(5, 3))
for i, T in enumerate(temperatures):
    rects = ax.bar(x + i * bar_width, scaled_probas[i], bar_width, label=f'Температура = {T}')

ax.set_ylabel('Вероятность')
ax.set_xticks(x)
ax.set_xticklabels(vocab.keys(), rotation=90)
ax.legend()

plt.tight_layout()
plt.savefig("temperature-plot.pdf")
plt.show()

- Мы видим, что перемасштабирование через температуру 0.1 приводит к более острому распределению, приближающемуся к `torch.argmax`, так что наиболее вероятное слово выбирается почти всегда:

In [ ]:
print_sampled_tokens(scaled_probas[1])

- Перемасштабированные вероятности через температуру 5 распределены более равномерно:

In [ ]:
print_sampled_tokens(scaled_probas[2])

- Если предположить, что на вход LLM подано "every effort moves you", использование описанного выше подхода может иногда приводить к бессмысленным текстам, таким как "every effort moves you pizza", в 3,2% случаев (32 раза из 1000)

 ---
 &nbsp;
## Упражнение 5.1. Использование функции print_sampled_tokens

- Мы можем вывести количество раз, которое слово "pizza" было сэмплировано, используя функцию `print_sampled_tokens`, которую мы определили в этом разделе
- Давайте начнём с кода, который мы определили в разделе 5.3.1

- Оно сэмплируется 0 раз, если температура равна 0 или 0.1, и сэмплируется 32 раза, если температура увеличена до 5. Оценённая вероятность составляет 32/1000 * 100% = 3.2%

- Фактическая вероятность равна 4.3% и содержится в перемасштабированном тензоре softmax-вероятностей (`scaled_probas[2][6]`)

- Проитерируемся по `scaled_probas` и выведем частоты сэмплирования в каждом случае:

In [ ]:
for i, probas in enumerate(scaled_probas):
    print("\n\nТемпература:", temperatures[i])
    print_sampled_tokens(probas)

- Обратите внимание, что сэмплирование даёт приблизительную оценку фактических вероятностей, когда сэмплируется слово "pizza"
- Например, если оно сэмплируется 32/1000 раз, оценённая вероятность составляет 3,2%
- Чтобы получить фактическую вероятность, мы можем проверить вероятности напрямую, обратившись к соответствующей записи в `scaled_probas`

- Поскольку "pizza" является 7-й записью в словаре, для температуры 5 мы получаем её следующим образом:

In [ ]:
temp5_idx = 2
pizza_idx = 6

scaled_probas[temp5_idx][pizza_idx]

Существует вероятность 4,3%, что слово "pizza" будет сэмплировано, если температура установлена на 5

---

&nbsp;
### 5.3.2. Выборка по принципу top-k

- Чтобы иметь возможность использовать более высокие температуры для увеличения разнообразия вывода и снижения вероятности появления бессмысленных предложений, мы можем ограничить выборку токенов k наиболее вероятными токенами (top-k):

<img src="https://camo.githubusercontent.com/7bc113d7a12a49a473a9059f425850564ce392aa34d8925e6edeb0a9558b73cb/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f31352e77656270" width=800px>

- (Обратите внимание, что числа на этом рисунке округлены до двух знаков после запятой, чтобы уменьшить визуальный шум. Значения в строке Softmax в сумме должны давать 1.0.)

- В коде это можно реализовать следующим образом:

In [ ]:
top_k = 3
top_logits, top_pos = torch.topk(next_token_logits, top_k)

print("Лучшие логиты:", top_logits)
print("Лучшие позиции:", top_pos)

 ---

### Что делает `torch.topk(next_token_logits, top_k)` «под капотом»

Эта функция решает задачу: **найти `k` самых больших значений в тензоре и вернуть как сами значения, так и их индексы.**

Представь, что `next_token_logits` — это одномерный тензор (вектор) оценок для каждого токена в словаре, например:

```
next_token_logits = [1.2, 0.8, 3.5, 0.2, 2.7, 1.9, ...]  (длина = размер словаря, например 50000)
```

**1. Поиск k-го наибольшего значения (k-й порядковой статистики)**

Внутри используется быстрый алгоритм частичной сортировки (обычно на основе `torch.sort` или специализированных CUDA-ядер, оптимизированных для поиска top-k без полной сортировки). Алгоритм не сортирует весь массив целиком, а находит только k лучших элементов, что гораздо эффективнее (сложность примерно O(n + k log k) вместо O(n log n)).

**2. Разделение результата на две части**

- **`top_logits`** — вектор длины `k`, содержащий **сами значения** от наибольшего к наименьшему.
- **`top_pos`** — вектор длины `k`, содержащий **индексы (позиции)** этих значений в исходном тензоре, в том же порядке.

**3. Конкретный пример**

Допустим, `next_token_logits` выглядит так (реально он длиннее, но для примера):

```python
next_token_logits = torch.tensor([1.2, 0.8, 3.5, 0.2, 2.7])
                                    #  0    1    2    3    4   <- индексы
```

Вызов `torch.topk(next_token_logits, 3)` выдаст:

- `top_logits  = [3.5, 2.7, 1.2]`   ← сами значения
- `top_pos     = [2,   4,   0  ]`   ← их позиции в исходном тензоре

- 3.5 — самое большое, находится на позиции 2
- 2.7 — второе по величине, на позиции 4
- 1.2 — третье, на позиции 0

### Зачем это в языковых моделях

Это ключевой шаг в стратегиях декодирования **Top-K сэмплирования**:

- Модель оставляет только `k` токенов с самыми высокими оценками.
- Остальные токены «зануляются» (их вероятности становятся равными 0).
- Затем из этих `k` лучших выбирается следующий токен случайным образом.

Это помогает избежать слишком предсказуемых повторяющихся ответов и при этом отсекает откровенно плохие варианты.

 ---

In [ ]:
new_logits = torch.where(
    condition=next_token_logits < top_logits[-1],
    input=torch.tensor(float("-inf")), 
    other=next_token_logits
)

print(new_logits)

> ПРИМЕЧАНИЕ:
>
> Альтернативная, немного более эффективная реализация предыдущей ячейки кода выглядит следующим образом:
>
> ```python
> new_logits = torch.full_like( # создать тензор, заполненный значениями -inf
>    next_token_logits, -torch.inf
>)   
> new_logits[top_pos] = next_token_logits[top_pos] # скопировать top-k значений в тензор с -inf
> ```
> <br>
> Подробнее см. https://github.com/rasbt/LLMs-from-scratch/discussions/326

In [ ]:
topk_probas = torch.softmax(new_logits, dim=0)
print(topk_probas)

&nbsp;
### 5.3.3 Изменение функции генерации текста

- В предыдущих двух подразделах были представлены сэмплирование с температурой и сэмплирование top-k
- Давайте используем эти две концепции, чтобы модифицировать функцию `generate_text_simple` и создать новую функцию `generate`:

In [ ]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):

    # Цикл for такой же, как и раньше: получаем логиты и фокусируемся только на последнем временном шаге
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # Новое: фильтрация логитов с помощью сэмплирования top_k
        if top_k is not None:
            # Оставляем только top_k значений
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        # Новое: применение температурного масштабирования
        if temperature > 0.0:
            logits = logits / temperature

            # Новое (отсутствует в книге): совет по численной стабильности для получения эквивалентных результатов на устройстве mps
            # вычитаем максимум по строкам перед softmax
            logits = logits - logits.max(dim=-1, keepdim=True).values
            
            # Применяем softmax для получения вероятностей
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

            # Сэмплируем из распределения
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

        # Иначе — как раньше: получаем индекс элемента словаря с наибольшим значением логита
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)

        if idx_next == eos_id:  # Останавливаем генерацию досрочно, если встретился токен конца последовательности и указан eos_id
            break

        # Как и раньше: добавляем сэмплированный индекс к текущей последовательности
        idx = torch.cat((idx, idx_next), dim=1)  # (batch_size, num_tokens+1)

    return idx

 ---

### Код выше


### Сигнатура функции

```python
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
```
Создаём функцию `generate`, которая принимает: саму модель, начальный текст, сколько токенов генерировать, размер памяти, температуру, фильтр top-k и стоп-сигнал.

### Цикл генерации
```python
for _ in range(max_new_tokens):
```
Повторяем генерацию столько раз, сколько токенов попросили создать. Каждый повтор — один новый токен.

### Обрезаем контекст
```python
idx_cond = idx[:, -context_size:]
```
Оставляем только последние токены, которые помещаются в память модели.
**Пример:** Если `context_size=1024`, а разговор длиной 2000 токенов — берём только последние 1024.

### Генерация логитов
```python
with torch.no_grad():
```
Говорим PyTorch: "Не запоминай, как мы считали, нам не нужны градиенты". Это экономит память при генерации текста.

```python
logits = model(idx_cond)
```
Модель смотрит на контекст и выдаёт оценки для каждого токена в словаре. Каждый токен получает число — насколько он хорош как продолжение.»

```python
logits = logits[:, -1, :]
```
Из всех предсказаний модели берём только последнее — оценку для СЛЕДУЮЩЕГО токена. Предсказания для прошлых токенов нам не нужны.

### Top-K фильтрация
```python
if top_k is not None:
```
Если пользователь включил фильтр top-k (например, топ-50), то применяем его.

```python
top_logits, _ = torch.topk(logits, top_k)
```
Находим `top_k` лучших оценок и их индексы. Нижнее подчёркивание значит "индексы нам не нужны, только значения".

```python
min_val = top_logits[:, -1]
```
Берём САМУЮ МАЛЕНЬКУЮ из лучших оценок — это наш ПОРОГ. Всё, что ниже этого порога, мы убьём.

```python
logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)
```
Для КАЖДОГО токена проверяем: если его оценка ниже порога — заменяем на минус бесконечность (она потом превратится в ноль вероятности). Если выше — оставляем как есть.

### Температурное масштабирование
```python
if temperature > 0.0:
```
Если температура задана (и она больше нуля), то применяем творческий режим.

```python
logits = logits / temperature
```
ДЕЛИМ ВСЕ оценки на температуру. Температура 2.0 делает всех "равнее" (больше случайности), температура 0.5 делает фаворитов ещё сильнее (меньше случайности).

### Численная стабильность
```python
logits = logits - logits.max(dim=-1, keepdim=True).values
```
Вычитаем самую большую оценку из всех оценок. Это трюк, чтобы компьютер не взорвался от огромных чисел при вычислении softmax. Представьте: если есть оценка 1000, то `e^1000` — это бесконечность для компьютера. Вычитание максимума спасает.

### Превращение в вероятности
```python
probs = torch.softmax(logits, dim=-1)
```
Превращаем оценки в ПРОЦЕНТЫ. Softmax делает так, что все числа становятся от 0 до 1 и в сумме дают 100%. Токены с минус бесконечностью получают 0% вероятности.

### Случайный выбор
```python
idx_next = torch.multinomial(probs, num_samples=1)
```
Тянем жребий! Токен с вероятностью 70% выпадет в 70% случаев, с 20% — в 20% случаев. Это как бросать кубик, где грани разного размера.

### Жадный выбор (без температуры)
```python
else:
    idx_next = torch.argmax(logits, dim=-1, keepdim=True)
```
Если температура не задана — просто бери токен с САМОЙ ВЫСОКОЙ оценкой. Всегда одно и то же. Скучно, но предсказуемо.

### Проверка стоп-сигнала
```python
if idx_next == eos_id:
    break
```
Если выпал токен "конец текста" и мы знаем его ID — стоп машина! Дальше не генерируем.

### Сборка результата
```python
idx = torch.cat((idx, idx_next), dim=1)
```
Приклеиваем новый токен справа ко всем предыдущим. Была цепочка из 100 токенов — стала из 101. Идём на следующий круг.

### Возврат результата
```python
return idx
```
Отдаём пользователю полную цепочку токенов (начало + всё, что сгенерировали). Потом их превратят обратно в человеческий текст.

 ---

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(inference_device),
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=25,
    temperature=1.4
)

print("Вывод:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
torch.save(model.state_dict(), "model.pth")  # Сохраняет ВСЕ знания обученной модели в один файл на диске

 ---

 &nbsp;
## Упражнение 5.2. Различные значения температуры и параметры top-k

- Как температура, так и настройки top-k необходимо подбирать индивидуально для каждой LLM (своего рода процесс проб и ошибок, пока не будут получены желаемые результаты)
- При этом желаемые результаты также зависят от конкретного приложения:
  - Более низкие значения top-k и температуры приводят к менее случайным результатам, что предпочтительно при создании образовательного контента, технической документации, ответов на вопросы, анализа данных, генерации кода и т.п.
  - Более высокие значения top-k и температуры приводят к более разнообразным и случайным результатам, что больше подходит для задач мозгового штурма, творческого письма и т.п.

---


## Упражнение 5.3. Настройки для функции generate

Существует несколько способов обеспечить детерминированное поведение функции `generate`:

1. Установка `temperature=0.0`;
2. Установка `top_k=1`.

Ниже приведён автономный пример с использованием кода из главы 5:

In [ ]:
import tiktoken
import torch
from previous_chapters import GPTModel


GPT_CONFIG_124M = {
    "vocab_size": 50257,  # Размер словаря
    "context_length": 256,       # Укороченная длина контекста (исходная: 1024)
    "emb_dim": 768,       # Размерность эмбеддингов
    "n_heads": 12,        # Количество голов внимания
    "n_layers": 12,       # Количество слоёв
    "drop_rate": 0.1,     # Коэффициент дропаута
    "qkv_bias": False     # Смещение query-key-value
}


torch.manual_seed(123)

tokenizer = tiktoken.get_encoding("gpt2")
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(torch.load("model.pth", weights_only=True))
model.eval();  # Режим оценки/генерации (eval = evaluation)

In [ ]:
pip install tqdm

In [ ]:
from gpt_generate import generate, text_to_token_ids, token_ids_to_text
from previous_chapters import generate_text_simple

In [ ]:
# Детерминированная функция, использующая torch.argmax

start_context = "Every effort moves you"

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Вывод:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
# Детерминированное поведение: без top_k, без температурного масштабирования

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=None,
    temperature=0.0
)

print("Вывод:\n", token_ids_to_text(token_ids, tokenizer))

- Обратите внимание, что повторное выполнение предыдущей ячейки кода приведёт к генерации точно такого же текста:

In [ ]:
# Детерминированное поведение: без top_k, без температурного масштабирования

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=None,
    temperature=0.0
)

print("Вывод:\n", token_ids_to_text(token_ids, tokenizer))

 ---

&nbsp;
## 5.4. Загрузка и сохранение весов модели в PyTorch

- Обучение LLM требует больших вычислительных затрат, поэтому крайне важно уметь сохранять и загружать веса LLM

<img src="https://camo.githubusercontent.com/8ed6c344e305bf119c32571d13ca00771db368ad79ff1d6d901464c668de75bc/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f31362e77656270" width=800px>

- Рекомендуемый способ в PyTorch — сохранять веса модели, так называемый `state_dict`, применяя функцию `torch.save` к методу `.state_dict()`:

In [ ]:
torch.save(model.state_dict(), "model.pth")

- Затем мы можем загрузить веса модели в новый экземпляр `GPTModel` следующим образом:

In [ ]:
model = GPTModel(GPT_CONFIG_124M)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Используйте PyTorch 2.9 или новее для стабильной работы с mps
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Устройство:", device)

model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))
model.eval();

- Для обучения LLM обычно используются адаптивные оптимизаторы, такие как Adam или AdamW, вместо обычного SGD
- Эти адаптивные оптимизаторы хранят дополнительные параметры для каждого веса модели, поэтому имеет смысл сохранять и их, если мы планируем продолжить предварительное обучение позже:

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    }, 
    "model_and_optimizer.pth"
)

In [ ]:
checkpoint = torch.load("model_and_optimizer.pth", weights_only=True)

model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train();

 ---
 
&nbsp;
## Упражнение 5.4. Еще одна эпоха обучения

- В этой новой среде кода потребуется несколько дополнительных шагов, чтобы сделать процесс воспроизводимым
- Сначала мы загружаем токенизатор, модель и оптимизатор:


In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Размер словаря
    "context_length": 256, # Укороченная длина контекста (изначально: 1024)
    "emb_dim": 768,        # Размерность эмбеддингов
    "n_heads": 12,         # Количество голов внимания
    "n_layers": 12,        # Количество слоёв
    "drop_rate": 0.1,      # Коэффициент дропаута
    "qkv_bias": False      # Смещение query-key-value
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = tiktoken.get_encoding("gpt2")

checkpoint = torch.load("model_and_optimizer.pth", weights_only=True)
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train();

- Далее мы инициализируем загрузчик данных:

In [ ]:
file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

# Изначально в книге использовался код ниже.
# Однако urllib использует старые настройки протокола,
# что может вызвать проблемы у некоторых читателей,
# использующих VPN. Версия с `requests` выше
# более надёжна в этом отношении.

"""
import urllib.request

if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode('utf-8')
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()
"""


# Соотношение обучающей и валидационной выборок
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

- Наконец, мы используем функцию `train_model_simple` для обучения модели:

In [ ]:
num_epochs = 1
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

&nbsp;
## 5.5. Загрузка предварительно обученных весов от OpenAI

- Ранее мы обучили небольшую GPT-2 модель, используя для образовательных целей очень маленький сборник рассказов
- К счастью, нам не нужно тратить десятки и сотни тысяч долларов на предобучение модели на большом корпусе — мы можем загрузить предобученные веса, предоставленные OpenAI

---

---


⚠️ **Примечание: Некоторые пользователи могут столкнуться с проблемами в этом разделе из-за несовместимости с TensorFlow, особенно на определённых системах Windows. TensorFlow требуется здесь только для загрузки оригинальных файлов весов OpenAI GPT-2, которые мы затем конвертируем в PyTorch.
Если у вас возникли проблемы, связанные с TensorFlow, вы можете использовать альтернативный код ниже вместо оставшейся части кода в этом разделе.
Эта альтернатива основана на предварительно сконвертированных весах PyTorch, созданных с использованием того же процесса конвертации, описанного в предыдущем разделе. Подробности смотрите в ноутбуке:
[../02_alternative_weight_loading/weight-loading-pytorch.ipynb](../02_alternative_weight_loading/weight-loading-pytorch.ipynb).**

```python
file_name = "gpt2-small-124M.pth"
# file_name = "gpt2-medium-355M.pth"
# file_name = "gpt2-large-774M.pth"
# file_name = "gpt2-xl-1558M.pth"

url = f"https://huggingface.co/rasbt/gpt2-from-scratch-pytorch/resolve/main/{file_name}"

if not os.path.exists(file_name):
    urllib.request.urlretrieve(url, file_name)
    print(f"Загружено в {file_name}")

gpt = GPTModel(BASE_CONFIG)
gpt.load_state_dict(torch.load(file_name, weights_only=True))
gpt.eval()

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Используйте PyTorch 2.9 или новее для стабильной работы с mps
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
else:
    device = torch.device("cpu")
gpt.to(device);


torch.manual_seed(123)

token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)

print("Сгенерированный текст:\n", token_ids_to_text(token_ids, tokenizer))
```

---

---

- Сначала немного шаблонного кода для загрузки файлов от OpenAI и загрузки весов в Python
- Поскольку OpenAI использовала [TensorFlow](https://www.tensorflow.org/), нам потребуется установить и использовать TensorFlow для загрузки весов; [tqdm](https://github.com/tqdm/tqdm) — это библиотека прогресс-баров
- Раскомментируйте и запустите следующую ячейку, чтобы установить необходимые библиотеки

In [ ]:
# pip install tensorflow tqdm

In [ ]:
print("Версия TensorFlow:", version("tensorflow"))
print("Версия tqdm:", version("tqdm"))

In [ ]:
# Относительный импорт из gpt_download.py, находящегося в этой папке

from gpt_download import download_and_load_gpt2
# Альтернативный вариант:
# from llms_from_scratch.ch05 import download_and_load_gpt2

---

**Примечание**

- В очень редких случаях ячейка кода выше может вызвать ошибку `zsh: illegal hardware instruction python`, что может быть связано с проблемой установки TensorFlow на вашем компьютере
- Один из читателей обнаружил, что установка TensorFlow через `conda` решила проблему в этом конкретном случае, как упоминается [здесь](https://github.com/rasbt/LLMs-from-scratch/discussions/273#discussioncomment-12367888)
- Дополнительные инструкции вы можете найти в этом дополнительном [руководстве по настройке Python](https://github.com/rasbt/LLMs-from-scratch/tree/main/setup/01_optional-python-setup-preferences#option-2-using-conda)

---

- Затем мы можем загрузить веса модели для версии на 124 миллиона параметров следующим образом:

In [ ]:
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

In [ ]:
print("Настройки:", settings)

In [ ]:
print("Ключи параметров словаря:", params.keys())

In [ ]:
print(params["wte"])
print("Веса слоя вложения токенов:", params["wte"].shape)

- Альтернативно, "355M", "774M" и "1558M" также поддерживаются в качестве аргументов `model_size`
- Различия между этими моделями разных размеров представлены на рисунке ниже:

<img src="https://camo.githubusercontent.com/5435c19ef837e8a269b9f829af032ce0103519a6bd3d1030abdda58a6db498a1/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f31372e77656270" width=800px>

- Выше мы загрузили веса GPT-2 модели на 124M в Python, однако нам всё ещё нужно перенести их в наш экземпляр `GPTModel`
- Сначала мы инициализируем новый экземпляр GPTModel
- Обратите внимание, что оригинальная GPT-модель инициализировала линейные слои для матриц запроса, ключа и значения в модуле многоголового внимания с векторами смещения, что не является обязательным или рекомендуемым; однако, чтобы иметь возможность загрузить веса корректно, мы должны включить их, также установив `qkv_bias` в `True` в нашей реализации
- Мы также используем длину контекста в `1024` токена, которая использовалась в оригинальной модели (моделях) GPT-2

In [ ]:
# Определяем конфигурации моделей в словаре для компактности
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# Копируем базовую конфигурацию и обновляем её настройками конкретной модели
model_name = "gpt2-small (124M)"  # Пример названия модели
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

gpt = GPTModel(NEW_CONFIG)
gpt.eval();

- Следующая задача — присвоить веса OpenAI соответствующим тензорам весов в нашем экземпляре `GPTModel`

In [ ]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Несовпадение формы. Левая: {left.shape}, Правая: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

In [ ]:
import numpy as np

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])
    
    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight, 
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias, 
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight, 
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias, 
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight, 
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias, 
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale, 
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift, 
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale, 
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift, 
            params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])
    
    
load_weights_into_gpt(gpt, params)
gpt.to(device);

 ---

 ### Код выше

Этот код **переносит предобученные веса OpenAI GPT-2 в нашу собственную модель**, преобразуя форматы тензоров на лету.

**Что конкретно делает:**

1. **Копирует эмбеддинги**: Загружает веса позиционных (`wpe`) и токенных (`wte`) эмбеддингов

2. **Для каждого слоя (блока) модели**:
   - **Расщепляет объединённые веса внимания**: OpenAI хранит Q, K, V в одной матрице (`c_attn`), а у нас они разделены — поэтому `np.split` делит матрицу на 3 части
   - **Транспонирует**: OpenAI использует другой порядок осей (строки вместо столбцов), поэтому применяется `.T`
   - **Загружает все компоненты трансформера**: веса и смещения для внимания, скрытых слоёв (feed-forward), нормализации слоёв (LayerNorm)

3. **Загружает финальные слои**: выходную норму и проекцию в токены (out_head)

Это как "переводчик форматов" между архитектурой OpenAI (TensorFlow) и нашей (PyTorch).

 ---

- Если модель загружена корректно, мы можем использовать её для генерации нового текста с помощью нашей предыдущей функции `generate`:

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)

print("Вывод:\n", token_ids_to_text(token_ids, tokenizer))

- Мы знаем, что загрузили веса модели правильно, потому что модель способна генерировать связный текст; если бы мы допустили хотя бы небольшую ошибку, модель не смогла бы этого сделать

 ---

 &nbsp;
## Упражнение 5.5. Расчет потерь

- Мы можем использовать следующий код для вычисления значений функции потерь модели GPT на обучающей и валидационной выборках:

```python
train_loss = calc_loss_loader(train_loader, gpt, device)
val_loss = calc_loss_loader(val_loader, gpt, device)
```

- Результаты потерь для модели на 124M параметров следующие:

```
Training loss: 3.754748503367106
Validation loss: 3.559617757797241
```

- Основное наблюдение заключается в том, что производительность на обучающей и валидационной выборках находится примерно на одном уровне
- Это может иметь несколько объяснений:

1. "The Verdict" не входил в набор данных для предобучения, когда OpenAI обучала GPT-2. Следовательно, модель явно не переобучается под обучающую выборку и работает одинаково хорошо как на обучающей, так и на валидационной частях "The Verdict". (Потери на валидационной выборке немного ниже, чем на обучающей, что нетипично для глубокого обучения. Однако это, вероятно, связано со случайным шумом, поскольку набор данных относительно мал. На практике, если переобучения нет, ожидается, что производительность на обучающей и валидационной выборках будет примерно одинаковой).

2. "The Verdict" входил в набор данных для обучения GPT-2. В этом случае мы не можем определить, переобучается ли модель на обучающих данных, потому что валидационная выборка также использовалась бы для обучения. Чтобы оценить степень переобучения, нам понадобился бы новый набор данных, созданный после того, как OpenAI завершила обучение GPT-2, чтобы гарантировать, что он не мог быть частью предобучения.

Приведённый ниже код представляет собой воспроизводимый автономный пример для этого нового ноутбука.

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Размер словаря
    "context_length": 256, # Укороченная длина контекста (изначально: 1024)
    "emb_dim": 768,        # Размерность эмбеддингов
    "n_heads": 12,         # Количество голов внимания
    "n_layers": 12,        # Количество слоёв
    "drop_rate": 0.1,      # Коэффициент дропаута
    "qkv_bias": False      # Смещение query-key-value
}


torch.manual_seed(123)

tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

In [ ]:
# Определяем конфигурации моделей в словаре для компактности
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# Копируем базовую конфигурацию и обновляем её настройками конкретной модели
model_name = "gpt2-small (124M)"  # Пример названия модели
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

gpt = GPTModel(NEW_CONFIG)
gpt.eval();

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
load_weights_into_gpt(gpt, params)
gpt.to(device);

In [ ]:
file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode('utf-8')
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()


# Соотношение обучающей и валидационной выборок
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [ ]:
torch.manual_seed(123) # Для воспроизводимости из-за перемешивания в загрузчике данных
train_loss = calc_loss_loader(train_loader, gpt, device)
val_loss = calc_loss_loader(val_loader, gpt, device)

print("Потери на обучении:", train_loss)
print("Потери на валидации:", val_loss)

Мы также можем повторить это для самой большой модели GPT-2, но не забудьте обновить длину контекста:

In [ ]:
settings, params = download_and_load_gpt2(model_size="1558M", models_dir="gpt2")

model_name = "gpt2-xl (1558M)"
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

gpt = GPTModel(NEW_CONFIG)
gpt.eval()

load_weights_into_gpt(gpt, params)
gpt.to(device)

torch.manual_seed(123)
train_loss = calc_loss_loader(train_loader, gpt, device)
val_loss = calc_loss_loader(val_loader, gpt, device)

print("Потери на обучении:", train_loss)
print("Потери на валидации:", val_loss)

 ---

## Упражнение 5.6. Эксперименты с моделями и сравнение текстов


- В основной главе мы экспериментировали с самой маленькой моделью GPT-2, которая имеет всего 124M параметров
- Причина заключалась в том, чтобы максимально снизить требования к ресурсам
- Однако вы можете легко экспериментировать с более крупными моделями, внеся минимальные изменения в код
- Например, вместо загрузки модели на 124M, чтобы загрузить модель на 1558M, нужно изменить всего 2 строки кода:

```python
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")
model_name = "gpt2-small (124M)"
```

- Обновлённый код становится таким:

```python
settings, params = download_and_load_gpt2(model_size="1558M", models_dir="gpt2")
model_name = "gpt2-xl (1558M)"
```

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Размер словаря
    "context_length": 256, # Укороченная длина контекста (изначально: 1024)
    "emb_dim": 768,        # Размерность эмбеддингов
    "n_heads": 12,         # Количество голов внимания
    "n_layers": 12,        # Количество слоёв
    "drop_rate": 0.1,      # Коэффициент дропаута
    "qkv_bias": False      # Смещение query-key-value
}


tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
from gpt_download import download_and_load_gpt2
from gpt_generate import load_weights_into_gpt


model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

model_name = "gpt2-xl (1558M)"
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

gpt = GPTModel(NEW_CONFIG)
gpt.eval()

settings, params = download_and_load_gpt2(model_size="1558M", models_dir="gpt2")
load_weights_into_gpt(gpt, params)

In [ ]:
from gpt_generate import generate, text_to_token_ids, token_ids_to_text

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)

print("Вывод:\n", token_ids_to_text(token_ids, tokenizer))

 ---